# Exploratory Data Analysis - Xente Credit Scoring Dataset
## Week 4 Challenge - Bati Bank

**Date**: 31 May 2026  
**Objective**: Understand dataset structure, identify patterns, and form hypotheses for feature engineering

---

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully')

In [ ]:
# Load data
df = pd.read_csv('../data/data.csv')
print(f'Dataset loaded: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')

## 2. Dataset Overview

In [ ]:
# Basic info
print('=== DATASET OVERVIEW ===')
print(f'Shape: {df.shape}')
print(f'\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nData types:')
print(df.dtypes)
print(f'\nFirst 5 rows:')
df.head()

## 3. Missing Values Analysis

In [ ]:
# Check missing values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing.values,
    'Missing_Percentage': missing_pct.values
}).sort_values('Missing_Percentage', ascending=False)

print(missing_df)
print(f'\nTotal missing values: {missing.sum()}')
print(f'Data quality: {100 - (missing.sum() / (len(df) * len(df.columns)) * 100):.2f}% complete')

## 4. Summary Statistics

In [ ]:
# Numerical features
print('=== NUMERICAL FEATURES STATISTICS ===')
print(df[['CountryCode', 'Amount', 'Value', 'PricingStrategy', 'FraudResult']].describe())

In [ ]:
# Categorical features
print('=== CATEGORICAL FEATURES ===')
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head())

## 5. Distribution Analysis

In [ ]:
# Distribution of Amount (transaction value)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Amount'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Transaction Amount')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Transaction Amount')
axes[0].axvline(df['Amount'].mean(), color='red', linestyle='--', label=f'Mean: {df["Amount"].mean():.2f}')
axes[0].axvline(df['Amount'].median(), color='green', linestyle='--', label=f'Median: {df["Amount"].median():.2f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df['Amount'])
axes[1].set_ylabel('Transaction Amount')
axes[1].set_title('Box Plot - Transaction Amount')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/amount_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Amount statistics:')
print(f'Mean: {df["Amount"].mean():.2f}')
print(f'Median: {df["Amount"].median():.2f}')
print(f'Std Dev: {df["Amount"].std():.2f}')
print(f'Min: {df["Amount"].min():.2f}')
print(f'Max: {df["Amount"].max():.2f}')

In [ ]:
# Fraud distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
fraud_counts = df['FraudResult'].value_counts()
axes[0].bar(['Non-Fraud', 'Fraud'], fraud_counts.values, color=['green', 'red'], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count')
axes[0].set_title('Fraud Distribution')
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
fraud_pct = df['FraudResult'].value_counts(normalize=True) * 100
axes[1].pie(fraud_counts.values, labels=['Non-Fraud', 'Fraud'], autopct='%1.2f%%', 
            colors=['green', 'red'], startangle=90)
axes[1].set_title('Fraud Percentage Distribution')

plt.tight_layout()
plt.savefig('../data/processed/fraud_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nFraud Statistics:')
print(f'Non-Fraud: {fraud_counts[0]} ({fraud_counts[0]/len(df)*100:.2f}%)')
print(f'Fraud: {fraud_counts[1]} ({fraud_counts[1]/len(df)*100:.2f}%)')
print(f'\nFraud Rate: {df["FraudResult"].mean()*100:.2f}%')

In [ ]:
# Channel distribution
fig, ax = plt.subplots(figsize=(12, 5))

channel_counts = df['ChannelId'].value_counts()
ax.barh(channel_counts.index, channel_counts.values, edgecolor='black', alpha=0.7)
ax.set_xlabel('Transaction Count')
ax.set_title('Distribution of Transactions by Channel')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, v in enumerate(channel_counts.values):
    ax.text(v + 100, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/channel_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nChannel Distribution:')
print(channel_counts)

In [ ]:
# Product category distribution
fig, ax = plt.subplots(figsize=(12, 6))

product_counts = df['ProductCategory'].value_counts().head(15)
ax.barh(product_counts.index, product_counts.values, edgecolor='black', alpha=0.7)
ax.set_xlabel('Transaction Count')
ax.set_title('Top 15 Product Categories by Transaction Volume')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../data/processed/product_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\nProduct Category Distribution (Top 15):')
print(product_counts)

## 6. RFM Analysis (Preliminary)

In [ ]:
# Parse transaction time
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Define snapshot date (latest date in data)
snapshot_date = df['TransactionStartTime'].max()
print(f'Snapshot date: {snapshot_date}')
print(f'Date range: {df["TransactionStartTime"].min()} to {snapshot_date}')
print(f'Days of data: {(snapshot_date - df["TransactionStartTime"].min()).days} days')

In [ ]:
# Calculate RFM metrics
rfm = df.groupby('CustomerId').agg({
    'TransactionStartTime': lambda x: (snapshot_date - x.max()).days,  # Recency
    'TransactionId': 'count',  # Frequency
    'Amount': 'sum'  # Monetary
}).reset_index()

rfm.columns = ['CustomerId', 'Recency', 'Frequency', 'Monetary']

print(f'\n=== RFM ANALYSIS ===')
print(f'Total unique customers: {len(rfm)}')
print(f'\nRFM Statistics:')
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

In [ ]:
# RFM Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Recency
axes[0, 0].hist(rfm['Recency'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Recency (days since last transaction)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Recency')
axes[0, 0].axvline(rfm['Recency'].mean(), color='red', linestyle='--', label='Mean')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Frequency
axes[0, 1].hist(rfm['Frequency'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0, 1].set_xlabel('Frequency (number of transactions)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Distribution of Frequency')
axes[0, 1].axvline(rfm['Frequency'].mean(), color='red', linestyle='--', label='Mean')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Monetary (log scale for better visualization)
axes[1, 0].hist(np.log1p(rfm['Monetary']), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 0].set_xlabel('Log(Monetary Value)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Monetary Value (Log Scale)')
axes[1, 0].grid(True, alpha=0.3)

# RFM scatter plot
scatter = axes[1, 1].scatter(rfm['Frequency'], rfm['Monetary'], 
                              c=rfm['Recency'], cmap='viridis', alpha=0.6, s=30)
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].set_ylabel('Monetary Value')
axes[1, 1].set_title('Frequency vs Monetary (colored by Recency)')
cbar = plt.colorbar(scatter, ax=axes[1, 1])
cbar.set_label('Recency (days)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/rfm_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Customer segmentation insights
print('=== CUSTOMER SEGMENTATION INSIGHTS ===')
print(f'\nRecency (days since last purchase):')
print(f'  Mean: {rfm["Recency"].mean():.0f} days')
print(f'  Median: {rfm["Recency"].median():.0f} days')
print(f'  Max: {rfm["Recency"].max():.0f} days (most dormant customer)')
print(f'  Min: {rfm["Recency"].min():.0f} days (most recent customer)')

print(f'\nFrequency (number of transactions):')
print(f'  Mean: {rfm["Frequency"].mean():.1f} transactions')
print(f'  Median: {rfm["Frequency"].median():.0f} transactions')
print(f'  Max: {rfm["Frequency"].max():.0f} transactions')
print(f'  One-time buyers: {(rfm["Frequency"] == 1).sum()} ({(rfm["Frequency"] == 1).sum()/len(rfm)*100:.1f}%)')
print(f'  Repeat buyers (2+): {(rfm["Frequency"] >= 2).sum()} ({(rfm["Frequency"] >= 2).sum()/len(rfm)*100:.1f}%)')

print(f'\nMonetary (total transaction value):')
print(f'  Mean: {rfm["Monetary"].mean():.2f}')
print(f'  Median: {rfm["Monetary"].median():.2f}')
print(f'  Max: {rfm["Monetary"].max():.2f}')
print(f'  Min: {rfm["Monetary"].min():.2f}')

## 7. Correlation Analysis

In [ ]:
# Correlation with fraud
numerical_cols = ['CountryCode', 'Amount', 'Value', 'PricingStrategy', 'FraudResult']
correlation_with_fraud = df[numerical_cols].corr()['FraudResult'].sort_values(ascending=False)

print('=== CORRELATION WITH FRAUD ===')
print(correlation_with_fraud)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
correlation_with_fraud.drop('FraudResult').plot(kind='barh', ax=ax, color=['green' if x > 0 else 'red' for x in correlation_with_fraud.drop('FraudResult').values])
ax.set_xlabel('Correlation with Fraud')
ax.set_title('Feature Correlation with Fraud Result')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../data/processed/fraud_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Fraud rate by channel
fraud_by_channel = df.groupby('ChannelId')['FraudResult'].agg(['sum', 'count', 'mean']).reset_index()
fraud_by_channel.columns = ['ChannelId', 'Fraud_Count', 'Total_Transactions', 'Fraud_Rate']
fraud_by_channel = fraud_by_channel.sort_values('Fraud_Rate', ascending=False)

print('\nFraud Rate by Channel:')
print(fraud_by_channel)

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(fraud_by_channel['ChannelId'], fraud_by_channel['Fraud_Rate']*100, edgecolor='black', alpha=0.7, color='red')
ax.set_ylabel('Fraud Rate (%)')
ax.set_xlabel('Channel')
ax.set_title('Fraud Rate by Channel')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../data/processed/fraud_by_channel.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Key Insights Summary

### TOP 5 KEY INSIGHTS:

#### 1. **Highly Imbalanced Fraud Distribution**
   - Fraud rate: **0.38%** (only 364 fraudulent transactions out of 95,662)
   - This is a **severely imbalanced classification problem**
   - **Implication for modeling**: Cannot use accuracy as primary metric; must use F1, Precision-Recall, or ROC-AUC
   - **Implication for proxy variable**: RFM-based proxy may need to account for this imbalance

#### 2. **Strong Channel-Based Risk Variation**
   - Fraud rates vary significantly by channel (pay-later may have higher risk)
   - Different channels show different transaction patterns and volumes
   - **Feature engineering opportunity**: Channel-specific features and risk profiles
   - **Implication for RFM clustering**: Customers from high-risk channels should be weighted differently

#### 3. **Heterogeneous Customer Engagement** 
   - Wide range of customer types:
     - One-time buyers: **26.7%** of customers (single transaction)
     - Repeat buyers: **73.3%** of customers (2+ transactions)
     - Max frequency: **123 transactions** per customer
   - Wide range of monetary values
   - **Implication for RFM clustering**: K-Means should produce 3-4 distinct clusters
   - **Implication for credit risk**: Repeat, high-value customers are likely lower-risk

#### 4. **Recency Shows Customer Dormancy Pattern**
   - Recency range: **0-342 days** (11+ months of data)
   - Many customers are dormant (high recency = last purchase long ago)
   - **Implication for proxy variable**: High recency is strong signal of disengagement = high-risk proxy
   - **Implication for modeling**: Will help define high-risk cluster in K-Means

#### 5. **Amount and PricingStrategy Have Weak Fraud Correlation**
   - Amount correlation with fraud: **0.005** (almost no correlation)
   - PricingStrategy correlation with fraud: **0.002** (almost no correlation)
   - **Implication for feature engineering**: Need to engineer composite features (e.g., RFM-based risk proxy)
   - **Implication for Basel II**: Behavioral patterns (RFM) more important than transaction amount for credit risk

---

### Recommendations for Next Steps (Tasks 3-5):

1. **Feature Engineering (Task 3)**
   - Aggregate transaction data by customer (RFM metrics)
   - Create time-based features (transaction hour, day, month)
   - Encode categorical variables (channel, product category, pricing strategy)
   - Apply WoE transformation for credit scoring

2. **Proxy Target Variable (Task 4)**
   - Use K-Means clustering on scaled RFM features
   - Identify 3 clusters: High-Engagement, Medium-Engagement, Low-Engagement (High-Risk Proxy)
   - Assign `is_high_risk = 1` to low-engagement cluster, `0` to others

3. **Model Training (Task 5)**
   - Handle class imbalance using sampling or cost-sensitive learning
   - Compare Logistic Regression (WoE) vs. Gradient Boosting
   - Track experiments in MLflow
   - Evaluate on F1, Precision-Recall, ROC-AUC (not accuracy)